# Agent Orchestration: Routines, Tools, and Handoffs

### INFO 5940 · Assignment 4

In the earlier assignments, we sent messages to a model and used retrieved
information to help it answer questions. Here, we will give an assistant tools
it can request and let it pass a conversation to another assistant.

We will use a customer-service example throughout. You will talk to the assistant,
respond to its questions, and see how it handles requests for help, purchases,
and refunds. As the notebook progresses, we will add tool execution and handoffs
to the same basic conversation loop.

## How to Use This Notebook

Run Sections A–G in order. The code is complete. In the interactive conversations,
type your own messages at the `User:` prompt and enter `exit` when you are ready
to move on. Rerunning a conversation cell starts a fresh conversation.

There are five **Your observations** areas. Use them to describe what you tried,
what the assistant did, and what the code tells you about that behavior. The
suggested messages are starting points; use your own wording and follow up on the
answers you receive. A few sentences in each response area is enough.

| Section | What you will explore |
| --- | --- |
| A. Get started | Create the client |
| B. Give the assistant a routine | Have a conversation using instructions alone |
| C. Describe a tool | See how Python functions are described to the model |
| D. Run a conversation with tools | Follow tool requests and their results |
| E. Switch between agents | Pass conversation history from sales to refunds |
| F. Let an agent request a handoff | See when sales transfers a request |
| G. Try the complete conversation | Talk to triage, sales, and support |

Run the sections in order. Each section builds on the previous one, and later
examples replace some of the earlier code. Section B explains how to return to
an earlier example when you want to try it again.

The orders, refunds, item lookups, and human escalation are simulated. You will
be making real model calls, but these tools do not contact a store or move money.

## A. Get Started

Before we can build a conversation, we need a way for Python to send messages
to the model and receive its replies. We will use the same client throughout the
notebook. What changes from section to section is the information we send and
what our code does with the reply.

`OpenAI` is the client class from the OpenAI Python package. `client = OpenAI()`
creates the object we will use to send requests through the course gateway. It
reads your existing key and gateway address from the environment; creating it
does not make a model call.

`BaseModel` lets us group an agent's settings, `Optional` describes a value that
can be absent, and `json` lets us read arguments returned in a tool call.

In [ ]:
# Import the API client, classes for grouping settings, and JSON utilities.
# OpenAI sends requests, BaseModel defines our Agent and Response objects,
# and json reads tool arguments. Imports do not make model calls.
from openai import OpenAI
from pydantic import BaseModel
from typing import Optional
import json
from os import environ

# Create a client using the API key and gateway URL in the environment.
# The first model call happens later, when we use chat.completions.create().
client = OpenAI()


## B. Give the Assistant a Routine

The notion of a "routine" is not strictly defined, and instead meant to capture the idea of a set of steps. Concretely, let's define a routine to be a list of instructions in natural language (which we'll represent with a system prompt), along with the tools necessary to complete them.

Let's take a look at an example. Below, we've defined a routine for a customer service agent instructing it to triage the user issue, then either suggest a fix or provide a refund. We've also defined the necessary functions `execute_refund` and `look_up_item`. We can call this a customer service routine, agent, assistant, etc; however the idea itself is the same: a set of steps and the tools to execute them.

For now, we will define the functions but leave them disconnected from the model.
This first conversation lets us see what the instructions alone can do.

We will build a function called `run_full_turn` to handle one user message and
the assistant's response. At first it only calls the model. Later versions also
run tools and handle transfers. Running a new definition replaces the earlier
version that Python has been using.

Later cells redefine `run_full_turn` and some tools. If you revisit an earlier
section, rerun its definitions before its examples, then continue forward in order.
For example, to repeat the first conversation after reaching the handoff section,
rerun Section B's routine and conversation cells, then run the following sections
in order again when you move on.

In [ ]:
# Define the customer-support routine as one instruction string.
# The model is asked to follow these steps; Python does not enforce them.
# The \n characters separate the steps in the prompt.
system_message = (
    "You are a customer support agent for ACME Inc. "
    "Always answer in a sentence or less. "
    "Follow the following routine with the user:\n"
    "1. First, ask probing questions and understand the user's problem deeper.\n"
    " - unless the user has already provided a reason.\n"
    "2. Propose a fix (make one up).\n"
    "3. ONLY if not satisfied, offer a refund.\n"
    "4. If accepted, search for the ID and then execute refund."
    ""
)

# These tools are simulations: lookup always returns the same item ID,
# and refund prints a summary and returns success without moving money.
# Defining the functions does not yet make them available to the model.
def look_up_item(search_query):
    """Use to find item ID.
    Search query can be a description or keywords."""
    return "item_132612938"

def execute_refund(item_id, reason="not provided"):
    print("Summary:", item_id, reason)
    return "success"


### Have a conversation

The loop below takes your input, adds it to `messages`, and asks the model for a
reply. Each new request includes the earlier messages.

Start with a problem involving something you bought. You might say “The boots I
ordered don't fit.” Respond to the assistant naturally. You can give more detail,
reject a suggested fix, or ask for a refund. Watch how it follows the routine.

Type `exit` at the `User:` prompt to end the conversation. The cell remains running
while it waits for your messages. You can rerun it to try a different conversation.

In [ ]:
# Send the instructions and conversation, then save and display the reply.
# This version updates messages directly and does not expose any tools.
def run_full_turn(system_message, messages):
    response = client.chat.completions.create(
        model="openai.gpt-4o",
        messages=[{"role": "system", "content": system_message}] + messages,
    )
    message = response.choices[0].message
    messages.append(message)
    if message.content: print("Assistant:", message.content)
    return message

# Start a fresh conversation and keep accepting user messages until exit.
# Each call includes the earlier turns, so the assistant can use that history.
messages = []
while True:
    user = input("User: ")
    if user:
        print("User:", user)
    if user == "exit":
        break
    messages.append({"role": "user", "content": user})
    run_full_turn(system_message, messages)


### Your observations 1

Try expressing a problem in two ways: first with little detail, then with a clear
explanation of what went wrong and what you want. You can do this in one
conversation or start again by entering `exit` and rerunning the cell.

What did the assistant ask or suggest? Did it follow the routine when you asked
for a refund? If it said it had taken an action, could that action actually have
happened through this version of the code?

**Your response:**

*Describe the messages you tried and what happened here.*

## C. Describe a Tool

We have asked the assistant to look up items and issue refunds, but instructions
alone do not connect it to those Python functions. The next step is to tell the
model which functions it can request and what inputs each one expects.

To make a function available to the model, we send a **tool schema** describing
its name, purpose, parameters, and required arguments. The executable Python
function stays in our program.

The helper below builds that description from a function's name, docstring, type
annotations, and defaults. Read the comments and run it as supplied.

In [ ]:
# Build a tool schema by inspecting a function without executing it.
# The schema describes its name, docstring, parameter types, and required inputs.
import inspect

def function_to_schema(func) -> dict:

    # Map Python annotations to JSON schema types. Parameters without a
    # recognized annotation are described as strings by this simple helper.
    type_map = {
        str: "string",
        int: "integer",
        float: "number",
        bool: "boolean",
        list: "array",
        dict: "object",
        type(None): "null",
    }
    try:
        signature = inspect.signature(func)
    except ValueError as e:
        raise ValueError(
            f"Failed to get signature for function {func.__name__}: {str(e)}"
        )

    # Build one property for each parameter. Then collect the parameters
    # without defaults: those arguments must be supplied by the caller.
    parameters = {}
    for param in signature.parameters.values():
        try:
            param_type = type_map.get(param.annotation, "string")
        except KeyError as e:
            raise KeyError(
                f"Unknown type annotation {param.annotation} for parameter {param.name}: {str(e)}"
            )
        parameters[param.name] = {"type": param_type}
    required = [
        param.name
        for param in signature.parameters.values()
        if param.default == inspect._empty
    ]

    # Package the description in the API tool format. The docstring supplies
    # the description; the executable Python function stays in our program.
    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": (func.__doc__ or "").strip(),
            "parameters": {
                "type": "object",
                "properties": parameters,
                "required": required,
            },
        },
    }


### Inspect an example schema

Run the example below and compare the function definition with the printed
schema. Notice where the docstring appears and why `some_optional` is not in
`required`. Creating a schema does not run the function.

In [ ]:
# The third parameter has a type annotation; the last has a default.
# Pass the function itself to the helper, without calling it. The printed
# JSON shows how those details and the docstring become a tool description.
def sample_function(param_1, param_2, the_third_one: int, some_optional="John Smith"):
    """
    This is my docstring. Call this function when you want.
    """
    print("Hello, world")

schema =  function_to_schema(sample_function)
print(json.dumps(schema, indent=2))


### Inspect the refund tool's schema

Now apply the same helper to `execute_refund`, one of the functions our assistant
will actually request. Compare this schema with the function in Section B.
`item_id` is required, while `reason` has a default value and is optional.

The current function has no docstring or type annotations. Its description is
therefore empty, and this helper describes its parameters as strings. The schema
contains information about the function; it does not contain or run its code.

In [ ]:
# Describe the refund function without executing it.
schema_execute_refund = function_to_schema(execute_refund)
print(json.dumps(schema_execute_refund, indent=2))


## D. Run a Conversation with Tools

A tool schema tells the model what it can request. We still need Python to carry
out that request and give the result back. In this section, we will inspect those
steps separately before putting them into a conversation loop. This lets us see
the difference between requesting a refund and executing the refund function.

First, inspect one model request. We describe the lookup and refund functions
and ask the model to look up an item. The model may return a tool request rather
than an ordinary reply. Look for its function name, arguments, and call ID.

In [ ]:
# Start with a user request and describe the available Python functions.
# The model receives their schemas and chooses whether to request a tool.
# This API call does not execute any of those functions.
messages = [{"role": "user", "content": "Look up the black boot."}]
tools = [execute_refund, look_up_item]
tool_schemas = [function_to_schema(tool) for tool in tools]
response = client.chat.completions.create(
            model="openai.gpt-4o",
            messages=messages,
            tools=tool_schemas,
        )

# Save the assistant message before adding any tool results.
# Display its text, requested functions, arguments, and tool-call IDs.
message = response.choices[0].message
messages.append(message)
print(message.model_dump_json(indent=2))


### Inspect the tools sent to the model

`tool_schemas` is the list passed through `tools=` in the request above. Print it
to see the names, descriptions, and arguments available to the model. Compare the
lookup tool's description with the refund tool's description. These are the
schemas the model received, not the Python function bodies.

In [ ]:
# Show the complete list of tool descriptions supplied in the request.
print(json.dumps(tool_schemas, indent=2))


### Run the requested function

`tools_map` connects each tool name to its Python function. `json.loads` converts
the requested arguments into a dictionary, and `**args` passes those arguments
to the function.

The result is saved as a `tool` message. Its `tool_call_id` connects it to the
assistant's request. If the assistant did not request a tool, the loop has nothing
to execute. To repeat this example, rerun the request cell first.

In [ ]:
# Map each tool name to its Python function. To execute a request, read
# its JSON arguments into a dictionary and pass them using **args.
# For example, **{"search_query": "boots"} passes search_query="boots".
tools_map = {tool.__name__: tool for tool in tools}

def execute_tool_call(tool_call, tools_map):
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    print(f"Tool call requested by assistant: {name}({args})")
    return tools_map[name](**args)

# Run each requested function and save its result as a tool message.
# The tool_call_id pairs that result with the assistant request. Print the
# new dictionary directly; the rest of history can contain SDK objects.
for tool_call in message.tool_calls or []:
    result = execute_tool_call(tool_call, tools_map)
    result_message = {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result,
    }
    messages.append(result_message)
    print("Tool result:")
    print(json.dumps(result_message, indent=2))


### Connect the request to the Python function

`tools_map` is a dictionary of names and actual Python functions. The function
representations may include memory addresses; those addresses are not item IDs
or tool-call IDs.

The next cell prints that dictionary, followed by three fields for each requested
tool, in this order:

1. **Name:** which function the model requested. Python uses this name to look up
   the function in `tools_map`.
2. **Arguments:** a JSON string containing the inputs. `json.loads` turns it into
   the dictionary passed to the function.
3. **ID:** identifies this particular request. The result message repeats it as
   `tool_call_id`, so the model knows which request the result answers.

We loop over the current message's requests because there can be more than one.
If there are none, the cell prints only the function map.

In [ ]:
# Inspect the function map and each request's name, JSON arguments, and ID.
print(tools_map)
for tool_call in message.tool_calls or []:
    print(tool_call.function.name)
    print(tool_call.function.arguments)
    print(tool_call.id)


### Continue the conversation using the results

Now we put those steps into `run_full_turn`. A single user message may lead to
several model calls: the assistant can look up an item, use its ID to request a
refund, and then reply. Our code executes the requested functions and sends their
results back until the model returns without requesting a tool.

The conversation at the bottom works like Section B. Try the same customer
problem again and respond to the assistant's questions. This time, watch for
printed tool calls as well as replies. Enter `exit` when you are finished.

### Read the conversation output

The printed labels distinguish what is happening:

- **User:** the message you typed. We print it so it remains visible in the output.
- **Assistant:** the model's conversational reply.
- **Tool call:** the function and arguments requested by the model, which Python
  is about to execute.
- **Tool result:** the dictionary containing the function's returned text and the
  matching `tool_call_id`. The next model request includes this message.

A tool request belongs to an assistant message because the model produced it.
The tool's result is a separate message with `role="tool"`. The printed labels
help us follow the process; they do not change those API message roles.

In [ ]:
# Make the lookup and refund functions available to this conversation.
tools = [execute_refund, look_up_item]

# Process one user turn, which may require several model calls.
# Work on a copy of the history and return only the messages added here.
# The caller appends those messages to its saved conversation.
def run_full_turn(system_message, tools, messages):
    num_init_messages = len(messages)
    messages = messages.copy()
    while True:

        # Build tool descriptions for the model and a name-to-function map
        # for Python. The model receives schemas, not executable functions.
        tool_schemas = [function_to_schema(tool) for tool in tools]
        tools_map = {tool.__name__: tool for tool in tools}

        # Ask the model what to do next and save its assistant message.
        # Display any text. If it requests no tools, this turn is finished.
        response = client.chat.completions.create(
            model="openai.gpt-4o",
            messages=[{"role": "system", "content": system_message}] + messages,
            tools=tool_schemas or None,
        )
        message = response.choices[0].message
        messages.append(message)
        if message.content:
            print("Assistant:", message.content)
        if not message.tool_calls:
            break

        # Execute every requested function and save its result with the matching
        # tool-call ID. The next pass sends these results back to the model.
        for tool_call in message.tool_calls:
            result = execute_tool_call(tool_call, tools_map)
            result_message = {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            }
            messages.append(result_message)
            print("Tool result:")
            print(json.dumps(result_message, indent=2))
    return messages[num_init_messages:]

# Read the requested function name and decode its JSON arguments.
# **args passes the dictionary entries as named arguments to the function.
# This is where the requested Python function actually runs.
def execute_tool_call(tool_call, tools_map):
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    print(f"Tool call requested by assistant: {name}({args})")
    return tools_map[name](**args)

# Start a fresh conversation and read messages until exit.
# Save all returned replies and tool results so later turns can use them.
messages = []

while True:
    user = input("User: ")
    if user:
        print("User:", user)
    if user == "exit":
        break
    messages.append({"role": "user", "content": user})
    new_messages = run_full_turn(system_message, tools, messages)
    messages.extend(new_messages)


### Your observations 2

Try asking for a refund in your own words. You could ask directly, or first
explain a problem and see whether the assistant offers one. Continue the
conversation so it has a chance to ask questions and act.

What happened before it requested a tool? Identify a tool call and explain how
its arguments relate to something you said. How is this different from the
conversation in Section B? If no tool ran, describe what the assistant did instead.

**Your response:**

*Describe your conversation and the tool calls you observed here.*

## E. Switch Between Agents

Our support routine now works with tools. Suppose we also want to handle sales:
we could keep adding instructions to the same routine, but the assistant would
then have to manage several different tasks at once. Instead, we will separate
sales and refunds and give each assistant the instructions and tools it needs.
We will first choose the assistant ourselves so the switch is easy to follow.

As we add tasks, we can give each assistant its own instructions and tools.
An `Agent` groups those settings. Creating one does not start a conversation or
make a model call.

In [ ]:
# Group an agent's name, model, instructions, and available Python functions.
# Creating this object does not call the model. The loop reads its settings
# when it builds a request and turns its functions into tool schemas.
class Agent(BaseModel):
    name: str = "Agent"
    model: str = "openai.gpt-4o"
    instructions: str = "You are a helpful Agent"
    tools: list = []


The next version of `run_full_turn` reads the instructions, model, and tools from
an `Agent`, rather than taking the instructions and tools separately. The tool
loop still works the same way.

In [ ]:
# Process one user turn, which may require several model calls.
# Work on a copy of the history and return only the messages added here.
# The caller appends those messages to its saved conversation.
def run_full_turn(agent, messages):
    num_init_messages = len(messages)
    messages = messages.copy()
    while True:

        # Build tool descriptions for the model and a name-to-function map
        # for Python. The model receives schemas, not executable functions.
        tool_schemas = [function_to_schema(tool) for tool in agent.tools]
        tools_map = {tool.__name__: tool for tool in agent.tools}

        # Ask the model what to do next and save its assistant message.
        # Display any text. If it requests no tools, this turn is finished.
        response = client.chat.completions.create(
            model=agent.model,
            messages=[{"role": "system", "content": agent.instructions}] + messages,
            tools=tool_schemas or None,
        )
        message = response.choices[0].message
        messages.append(message)
        if message.content:
            print("Assistant:", message.content)
        if not message.tool_calls:
            break

        # Execute every requested function and save its result with the matching
        # tool-call ID. The next pass sends these results back to the model.
        for tool_call in message.tool_calls:
            result = execute_tool_call(tool_call, tools_map)
            result_message = {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            }
            messages.append(result_message)
            print("Tool result:")
            print(json.dumps(result_message, indent=2))
    return messages[num_init_messages:]

# Read the requested function name and decode its JSON arguments.
# **args passes the dictionary entries as named arguments to the function.
# This is where the requested Python function actually runs.
def execute_tool_call(tool_call, tools_map):
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    print(f"Tool call requested by assistant: {name}({args})")
    return tools_map[name](**args)


### Pass the conversation from sales to refunds

The example below creates two agents and sends each one a customer message.
Our Python code chooses which agent handles each turn. Notice that the refund
request does not repeat the product name: it relies on the saved history.

Run the example, then try changing the customer messages in `user_query`. These
are short, editable examples; the full interactive conversation returns in Section G.

In [ ]:
# Create separate sales and refund agents with their own tools.
# These functions return demonstration results without contacting a store.
def execute_refund(item_name):
    return "success"

refund_agent = Agent(
    name="Refund Agent",
    instructions="You are a refund agent. Help the user with refunds.",
    tools=[execute_refund],
)

def place_order(item_name):
    return "success"

sales_assistant = Agent(
    name="Sales Assistant",
    instructions="You are a sales assistant. Sell the user a product.",
    tools=[place_order],
)

# Start a conversation with sales and save the returned messages.
# Here response is a list, so extend adds each message to the history.
messages = []
user_query = "Place an order for a black boot."
print("User:", user_query)
messages.append({"role": "user", "content": user_query})
response = run_full_turn(sales_assistant, messages)
messages.extend(response)

# Our code explicitly selects refunds for the next turn. The same history
# contains the product name, even though this message only refers to it.
user_query = "Actually, I want a refund."
print("User:", user_query)
messages.append({"role": "user", "content": user_query})
response = run_full_turn(refund_agent, messages)
messages.extend(response)


### Your observations 3

Change the product in the purchase request. In the second message, refer to it
indirectly, for example “Actually, can I return that?” Rerun the cell.

Could the refund agent identify the product? Use the reply or tool arguments as
evidence. Which line of Python selected the refund agent, and what allowed it
to see the earlier sales conversation?

**Your response:**

*Describe your changes and what the refund agent did here.*

## F. Let an Agent Request a Handoff

Choosing the agent in Python works when we already know what the customer needs.
In a conversation, that need may only become clear from what the customer says.
Here, we will let the sales assistant request a transfer, then have our code carry
out the switch. This uses the same tool-calling mechanism we have already built.

In Section E, our code chose when to switch agents. Now we give sales a
`transfer_to_refunds` tool so the model can request that switch.

This function returns the refund agent object. Our loop will recognize it and
use that agent's instructions and tools for the next request, keeping the
conversation history.

In [ ]:
# A transfer tool returns an Agent object rather than a text result.
# The handoff loop recognizes that object and switches to it. Sales now
# has both an order tool and a way to transfer the conversation to refunds.
refund_agent = Agent(
    name="Refund Agent",
    instructions="You are a refund agent. Help the user with refunds.",
    tools=[execute_refund],
)

def transfer_to_refunds():
    """Transfer refund requests to the refund agent."""
    return refund_agent

sales_assistant = Agent(
    name="Sales Assistant",
    instructions="You are a sales assistant. Sell the user a product.",
    tools=[place_order,transfer_to_refunds],
)


`Response` is our own small class for returning the active agent and the new
messages at the end of a turn. It is unrelated to the OpenAI Responses API.

In [ ]:
# Our own return-value class, unrelated to the OpenAI Responses API.
# It carries the agent active at the end of a turn and the new messages.
# Optional means the agent field also accepts None.
class Response(BaseModel):
    agent: Optional[Agent]
    messages: list


### Handle a transfer

Most of our tools return text, such as `"success"` or an item ID. A transfer tool
works differently: `transfer_to_refunds()` returns the `refund_agent` object.
That object holds the refund agent's instructions, model, and available tools.
Returning it does not make a model call or start a separate conversation.

In the loop below, `result` holds whatever the requested Python function returned.
The check `if type(result) is Agent:` identifies a transfer. The line
`current_agent = result` then selects that returned agent for the next pass
through the loop.

We still need to send a result back for the transfer tool call. Instead of putting
the Python agent object into the conversation, the code replaces `result` with
text such as `"Transferred to Refund Agent. Adopt persona immediately."`. It saves
that text in a tool message with the matching `tool_call_id`.

On the next pass, the loop builds a request using the new agent's instructions,
model, and tools. The `messages` list is kept, so the new agent receives the
conversation that led to the transfer. The text announces the transfer; the
change to `current_agent` is what makes our code use the new agent's settings.

When the model finishes without requesting another tool, `run_full_turn` returns
a `Response` containing the final active agent and the new messages. In the
interactive conversation later, we save both so the next user message continues
with that agent and the updated history.

In [ ]:
# Process one user turn, which may require several model calls.
# Work on a copy of the history and return only the messages added here.
# The caller appends those messages to its saved conversation.
def run_full_turn(agent, messages):
    current_agent = agent
    num_init_messages = len(messages)
    messages = messages.copy()
    while True:

        # Build the active agent's tool schemas and a map back to Python functions.
        # After a handoff, the next pass uses the new agent's settings.
        tool_schemas = [function_to_schema(tool) for tool in current_agent.tools]
        tools = {tool.__name__: tool for tool in current_agent.tools}

        # Ask the model what to do next and save its assistant message.
        # Display any text. If it requests no tools, this turn is finished.
        response = client.chat.completions.create(
            model=current_agent.model,
            messages=[{"role": "system", "content": current_agent.instructions}]
            + messages,
            tools=tool_schemas or None,
        )
        message = response.choices[0].message
        messages.append(message)
        if message.content:
            print(f"{current_agent.name}:", message.content)
        if not message.tool_calls:
            break

        # Execute the requested functions. An Agent result selects the next
        # agent; convert that result to text for the conversation. Pair every
        # tool result with its request ID before making the next model call.
        for tool_call in message.tool_calls:
            result = execute_tool_call(tool_call, tools, current_agent.name)
            if type(result) is Agent:
                current_agent = result
                result = (
                    f"Transferred to {current_agent.name}. Adopt persona immediately."
                )
            result_message = {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            }
            messages.append(result_message)
            print("Tool result:")
            print(json.dumps(result_message, indent=2))
    return Response(agent=current_agent, messages=messages[num_init_messages:])

# Read the requested function name and decode its JSON arguments.
# **args passes the dictionary entries as named arguments to the function.
# This is where the requested Python function actually runs.
def execute_tool_call(tool_call, tools, agent_name):
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    print(f"Tool call requested by {agent_name}: {name}({args})")
    return tools[name](**args)


Run this request and follow the printed agent names and tool calls. It begins
with sales even though the customer asks for a refund.

In [ ]:
# Start with sales and a fresh history. Try different customer wording
# and watch for a transfer. The returned Response contains the final
# agent and this turn's messages; this example runs just one user turn.
messages = []
user_query = "Hi I need a refund"
print("User:", user_query)
messages.append({"role": "user", "content": user_query})
response = run_full_turn(sales_assistant, messages)


### Your observations 4

Try changing `user_query` to an indirect request, such as “I'm not happy with the
boots I bought. What can I do?” Compare it with a direct refund request. Rerun
the cell for each message; each run starts a fresh conversation with sales.

Did both messages lead to a handoff, or did one lead to a question or another
reply? Point to the printed transfer call if one occurred. Explain how this
switch differs from the manual switch in Section E.

**Your response:**

*Record the wording you tried and how the assistant handled it here.*

## G. Try the Complete Conversation

We can now combine routines, tools, and handoffs in one ongoing conversation.
A customer does not have to know which agent to ask for, and their needs can
change during the chat. This final example lets us explore how agents route
those requests and use the history when someone else takes over.

The final example adds a triage agent that directs customers to sales or issues
and repairs. Each specialist can return the conversation to triage if the topic
changes. Read the instructions and tools for each agent, then run the definitions.

Sales uses fictional ACME products. Its order tool asks for `y` or `n` before
printing the result. Support uses a sample item ID and a simulated refund.
Human escalation prints a report and returns a message; it does not contact anyone.

In [ ]:
# Simulate escalation by printing a report and returning text.
# This does not contact a person or end the notebook session.
def escalate_to_human(summary):
    """Only call this if explicitly asked to."""
    print("Escalating to human agent...")
    print("\n=== Escalation Report ===")
    print(f"Summary: {summary}")
    print("=========================\n")
    return "Demo escalation recorded. A human would take over here."

# Transfer functions return agent objects without making model calls.
# Run this whole cell before chatting so all referenced agents exist.
def transfer_to_sales_agent():
    """Use for anything sales or buying related."""
    return sales_agent

def transfer_to_issues_and_repairs():
    """Use for issues, repairs, or refunds."""
    return issues_and_repairs_agent

def transfer_back_to_triage():
    """Call this if the user brings up a topic outside of your purview,
    including escalating to human."""
    return triage_agent

# Triage gathers enough information to route the conversation. Its tools
# can transfer or escalate, but cannot place orders or issue refunds.
triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "You are a customer service bot for ACME Inc. "
        "Introduce yourself. Always be very brief. "
        "Gather information to direct the customer to the right department. "
        "But make your questions subtle and natural."
    ),
    tools=[transfer_to_sales_agent, transfer_to_issues_and_repairs, escalate_to_human],
)

# Display an order summary and ask the person running the notebook.
# Only y, after trimming spaces and converting to lowercase, proceeds.
# Every other answer cancels the simulated order.
def execute_order(product, price: int):
    """Price should be in USD."""
    print("\n\n=== Order Summary ===")
    print(f"Product: {product}")
    print(f"Price: ${price}")
    print("=================\n")
    confirm = input("Confirm order? y/n: ").strip().lower()
    if confirm == "y":
        print("Order execution successful!")
        return "Success"
    else:
        print("Order cancelled!")
        return "User cancelled order."

# Give sales its routine, the order function, and a route back to triage.
# A change of topic can lead it to request that transfer.
sales_agent = Agent(
    name="Sales Agent",
    instructions=(
        "You are a sales agent for ACME Inc. "
        "Always answer in a sentence or less. "
        "Follow the following routine with the user:\n"
        "1. Ask them about any problems in their life related to catching roadrunners.\n"
        "2. Casually mention one of ACME's crazy made-up products can help.\n"
        " - Don't mention price.\n"
        "3. Once the user is bought in, drop a ridiculous price.\n"
        "4. Only after everything, and if the user says yes, "
        "tell them a crazy caveat and execute their order.\n"
        ""
    ),
    tools=[execute_order, transfer_back_to_triage],
)

# Support uses a fixed sample item ID and a simulated refund.
# Unlike execute_order, execute_refund has no confirmation input.
# The requirement to get customer acceptance appears in the prompt below.
def look_up_item(search_query):
    """Use to find item ID.
    Search query can be a description or keywords."""
    item_id = "item_132612938"
    print("Found item:", item_id)
    return item_id

def execute_refund(item_id, reason="not provided"):
    print("\n\n=== Refund Summary ===")
    print(f"Item ID: {item_id}")
    print(f"Reason: {reason}")
    print("=================\n")
    print("Refund execution successful!")
    return "success"

# Support can look up items, issue simulated refunds, or return to triage.
# Its instructions describe when it should take each action.
issues_and_repairs_agent = Agent(
    name="Issues and Repairs Agent",
    instructions=(
        "You are a customer support agent for ACME Inc. "
        "Always answer in a sentence or less. "
        "Follow the following routine with the user:\n"
        "1. First, ask probing questions and understand the user's problem deeper.\n"
        " - unless the user has already provided a reason.\n"
        "2. Propose a fix (make one up).\n"
        "3. ONLY if not satisfied, offer a refund.\n"
        "4. If accepted, search for the ID and then execute refund."
        ""
    ),
    tools=[execute_refund, look_up_item, transfer_back_to_triage],
)


### Talk to the agents

Run the cell below and have a conversation in your own words. You could start
with a damaged product, ask about buying something, or ask to speak to a person.
Respond to what the assistant actually says, and watch the agent names and tools.

Try changing the subject while the conversation is still running. For example,
after discussing a purchase, mention a problem with an earlier order. Does the
current specialist continue, ask a question, or transfer you?

Enter `exit` at the `User:` prompt to finish. At `Confirm order? y/n:`, enter `y`
or `n` instead. Rerunning this cell resets the history and starts with triage.
If the tools keep running without a reply, use the notebook's stop button;
these simple loops have no automatic step limit.

In [ ]:
# Begin a fresh conversation with triage. Read user messages until exit,
# and handle any tool calls and handoffs before asking for the next input.
agent = triage_agent

messages = []

while True:
    user = input("User: ")
    if user:
        print("User:", user)
    if user == "exit":
        break
    messages.append({"role": "user", "content": user})

    # Keep both the agent that finished the turn and every new message.
    # The next user turn continues with that agent and the accumulated history.
    response = run_full_turn(agent, messages)
    agent = response.agent
    messages.extend(response.messages)


### Your observations 5

Describe one conversation in which you changed the subject or asked to be
transferred. Which agents handled it, and what did you say before a transfer
happened? Did the next agent use details you had already provided?

If you reached an order confirmation, describe what happened when you answered.
Then compare `execute_order` and `execute_refund`: where is confirmation enforced
in Python, and where does the routine rely on the model following instructions?

**Your response:**

*Use a few messages and outputs from your conversation to explain what happened.*

## Before You Finish

- Run the examples and try your own messages in the conversations.
- Complete all five **Your observations** response areas.
- Keep the outputs you discuss. If you rerun a cell, describe the earlier result
  in your response so your comparison is clear.
- Save your work in `workbook.ipynb`.

There is no single conversation you need to reproduce. Explain what happened in
your runs, including any questions, unexpected replies, or missing tool calls.